# Step 10b — Targeted ground_truth re-sample (top-up)

Fetches **100 fresh candidates per requested `(smell, polarity)`** for buckets that fell short of 50 confirmed-correct labels after manual validation of the Step 10 pool.

Fresh = the `(file_key, smell, polarity)` triple is **not** already in `data/10_ground_truth_sample.json`. Same universe, strict filter, ranking, and LOC-quartile selection as Step 10 (all imported from `dataset_utils.ground_truth_sampling`).

**Edit `REQUESTS` in the cell below, then run-all.** Output: `data/10b_ground_truth_resample.json` (+ `data/step10b_report.json`) — same schema as `10_ground_truth_sample.json`, with `evaluation_targets` carrying only the requested target(s).

Prereq: Step 09 / 09b / 10 outputs present (`9_ground_truth_candidates.csv`, `9b_loc_cap.json`, `step9_report.json`, `step9b_report.json`, `10_ground_truth_sample.json` (exclusion source), and the `downloaded_workflows/` YAML corpus).

In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 10b — starting: targeted ground_truth re-sample (top-up)...", flush=True)
"""Step 10b: Targeted re-sample top-up for deficient (smell, polarity) targets.

Context (2026-05-28):
  Manual validation of the Step 10 pool left some (smell, polarity) buckets
  short of the N_PER_POLARITY=50 confirmed-correct goal — NOT because Step 10
  under-delivered (it shipped the full 100/polarity for every smell) but because
  detector false-positives/negatives consumed the 2x oversample margin. Known
  shortfalls after annotation:
    UnpinnedActionVersion  negative  39/50 confirmed
    MissingJobTimeout                      negative  33/50 confirmed
    UnsafeDeployOnFork                     positive   8/50 confirmed

  This step fetches N_RESAMPLE_PER_TARGET=100 *fresh* candidates per requested
  (smell, polarity) — files whose (file_key, smell, polarity) triple is NOT
  already present in data/10_ground_truth_sample.json — so annotators can label more
  and reach 50 confirmed per bucket.

Identical to Step 10 (all primitives imported from dataset_utils.ground_truth_sampling):
  - candidate universe : data/9_ground_truth_candidates.csv (same integrity chain)
  - LOC cap            : data/9b_loc_cap.json (verified vs step9b_report.json)
  - strict filter      : every behavioral smell instance_count <= 1
  - positive           : file has the smell (instance_count == 1 under strict)
  - negative           : pure absence
  - ranking            : (total_unique_smells ASC, workflow_run_count DESC, file_key ASC)
  - selection          : LOC-quartile balanced (_select_loc_balanced)

Differs from Step 10:
  - Targeted: only the (smell, polarity) pairs in REQUESTS.
  - Exclusion: drops (file_key, smell, polarity) triples already present in the
    Step 10 ground_truth. A file may reappear for a *new* target.
  - evaluation_targets carries ONLY the requested target(s) (surgical top-up).
  - Shortfall tolerated: emit min(N, available_fresh), record the gap in report.

Inputs:
  data/9_ground_truth_candidates.csv     candidate universe (Step 09)
  data/9b_loc_cap.json             LOC cap (Step 09b)
  data/10_ground_truth_sample.json       prior ground_truth (exclusion source)
  data/downloaded_workflows/       cached YAML corpus

Outputs:
  data/10b_ground_truth_resample.json    file-based ground_truth, same schema as 10_ground_truth_sample
  data/step10b_report.json         per-target stats + sampling config
"""

import json
import statistics
from pathlib import Path

import polars as pl
from dataset_utils.ground_truth_sampling import (
    BEHAVIORAL_SMELL_ORDER,
    PRESENTATIONAL_SMELLS,
    SMELL_CATALOG,
    _build_entry,
    _build_sort_key,
    _passes_strict,
    _select_loc_balanced,
    read_yaml,
)

# -- Re-sample request set ----------------------------------------------------
# Edit this list, then run-all. Each entry:
#   {"pascal": <behavioral smell>, "polarity": "positive" | "negative"}
# The same smell may appear twice (once per polarity) if both buckets fell short.
REQUESTS: list[dict] = [
    {"pascal": "UnpinnedActionVersion", "polarity": "negative"},
    {"pascal": "MissingJobTimeout",                    "polarity": "negative"},
    {"pascal": "UnsafeDeployOnFork",                   "polarity": "positive"},
]

N_RESAMPLE_PER_TARGET = 100
"""Fresh candidates emitted per requested (smell, polarity). Priority 1..N in
primary-rank order. Matches Step 10's delivered pool; no further oversample."""

VALID_POLARITIES = ("positive", "negative")


def _validate_requests(requests: list[dict]) -> None:
    seen: set[tuple[str, str]] = set()
    for req in requests:
        pascal = req.get("pascal")
        polarity = req.get("polarity")
        if pascal not in SMELL_CATALOG:
            raise ValueError(f"unknown smell pascal: {pascal!r}")
        if pascal in PRESENTATIONAL_SMELLS:
            raise ValueError(f"{pascal} is presentational — not evaluated, cannot re-sample")
        if polarity not in VALID_POLARITIES:
            raise ValueError(f"polarity must be one of {VALID_POLARITIES}, got {polarity!r}")
        if (pascal, polarity) in seen:
            raise ValueError(f"duplicate request: ({pascal}, {polarity})")
        seen.add((pascal, polarity))


def resample_ground_truth(
    requests: list[dict],
    candidates_path: Path,
    prior_ground_truth_path: Path,
    output_path: Path,
    report_path: Path,
    n: int = N_RESAMPLE_PER_TARGET,
) -> dict:
    import sys as _sys
    try:
        _sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
    except NameError:
        pass  # Jupyter: __file__ undefined; dataset_utils on path via wheel install
    from dataset_utils.hashing import compute_file_hash
    from dataset_utils.integrity import assert_input_matches_report

    _validate_requests(requests)

    # --- integrity: candidate universe matches Step 09 report -----------------
    input_sha = assert_input_matches_report(
        input_path=candidates_path,
        prior_report_path=Path("data/step9_report.json"),
        label="step 10b input",
    )

    # --- LOC cap from Step 09b (verify chain vs step9b_report.json) -----------
    loc_cap_path = Path("data/9b_loc_cap.json")
    step9b_report_path = Path("data/step9b_report.json")
    if not loc_cap_path.exists() or not step9b_report_path.exists():
        msg = (
            f"step 10b: LOC cap inputs missing. "
            f"Expected {loc_cap_path} and {step9b_report_path}. Run Step 09b first."
        )
        raise FileNotFoundError(msg)
    loc_cap_sha = compute_file_hash(loc_cap_path)
    with step9b_report_path.open("r", encoding="utf-8") as _rf:
        expected_cap_sha = json.load(_rf).get("output_sha256")
    if expected_cap_sha and expected_cap_sha != loc_cap_sha:
        msg = (
            f"step 10b: data/9b_loc_cap.json SHA drift. "
            f"current={loc_cap_sha}, expected={expected_cap_sha}. Re-run Step 09b."
        )
        raise RuntimeError(msg)
    with loc_cap_path.open("r", encoding="utf-8") as _cf:
        loc_cap = int(json.load(_cf)["cap"])
    print(f"  LOC cap from Step 09b: {loc_cap} (sha {loc_cap_sha[:16]}...)", flush=True)

    # --- exclusion set: (file_key, pascal, polarity) already in Step 10 ground_truth
    prior_sha = compute_file_hash(prior_ground_truth_path)
    with prior_ground_truth_path.open("r", encoding="utf-8") as _pf:
        prior = json.load(_pf)
    excluded: set[tuple[str, str, str]] = set()
    for entry in prior:
        fk = entry["file_key"]
        for t in entry.get("evaluation_targets", []):
            excluded.add((fk, t["pascal"], t["polarity"]))
    print(
        f"  exclusion set: {len(excluded):,} (file,smell,polarity) triples "
        f"from {prior_ground_truth_path.name}",
        flush=True,
    )

    # --- load + LOC-cap the candidate universe --------------------------------
    print("  reading input CSV (may take a moment for large files)...", flush=True)
    df = pl.read_csv(candidates_path, encoding="utf-8-sig")
    for col in ("total_unique_smells", "file_line_count", "workflow_run_count"):
        if df[col].dtype == pl.Utf8:
            df = df.with_columns(pl.col(col).cast(pl.Int64, strict=False).fill_null(0))
    rows_before_cap = df.height
    df = df.filter(pl.col("file_line_count") <= loc_cap)
    rows_after_cap = df.height
    loc_cap_excluded = rows_before_cap - rows_after_cap
    print(
        f"  LOC cap applied: {loc_cap_excluded:,} rows dropped; {rows_after_cap:,} retained.",
        flush=True,
    )

    rows_by_key: dict[str, dict] = {}
    smells_by_key: dict[str, list[dict]] = {}
    for r in df.iter_rows(named=True):
        rows_by_key[r["file_key"]] = r
        smells_by_key[r["file_key"]] = json.loads(r["smells_json"])

    # --- strict pool + per-smell presence index -------------------------------
    strict_keys: list[str] = []
    rejected_strict = 0
    for fk, smells in smells_by_key.items():
        if _passes_strict(smells):
            strict_keys.append(fk)
        else:
            rejected_strict += 1
    strict_set = set(strict_keys)

    strict_smell_presence: dict[str, set[str]] = {s: set() for s in BEHAVIORAL_SMELL_ORDER}
    for fk in strict_keys:
        for s in smells_by_key[fk]:
            pascal = s.get("pascal")
            if pascal in strict_smell_presence:
                strict_smell_presence[pascal].add(fk)

    # --- per-request selection ------------------------------------------------
    per_target_stats: list[dict] = []
    # (pascal, polarity, [(priority, file_key), ...])
    selections: list[tuple[str, str, list[tuple[int, str]]]] = []

    for req in requests:
        pascal, polarity = req["pascal"], req["polarity"]
        present = strict_smell_presence[pascal]
        pool_keys = present if polarity == "positive" else (strict_set - present)
        # Drop files already used for this exact (smell, polarity) in Step 10.
        fresh_keys = [k for k in pool_keys if (k, pascal, polarity) not in excluded]
        candidates = [rows_by_key[k] for k in fresh_keys]
        candidates.sort(key=_build_sort_key)
        selected = _select_loc_balanced(candidates, n)

        locs = [r["file_line_count"] or 0 for r in selected]
        paper_id = SMELL_CATALOG[pascal][0]
        delivered = len(selected)
        shortfall = max(0, n - delivered)
        if shortfall:
            print(
                f"  WARNING {pascal} {polarity}: only {delivered} fresh strict "
                f"candidates (requested {n}, short {shortfall}).",
                flush=True,
            )
        per_target_stats.append({
            "paper_id": paper_id,
            "pascal": pascal,
            "polarity": polarity,
            "available_fresh": len(candidates),
            "delivered": delivered,
            "shortfall": shortfall,
            "loc_min": min(locs) if locs else None,
            "loc_max": max(locs) if locs else None,
            "loc_median": int(statistics.median(locs)) if locs else None,
        })
        selections.append(
            (pascal, polarity, [(i + 1, r["file_key"]) for i, r in enumerate(selected)])
        )

    # --- build deduplicated entries carrying ONLY the requested targets -------
    ground_truth_entries: dict[str, dict] = {}
    yaml_cache: dict[str, str | None] = {}
    skipped_no_yaml: list[str] = []

    for pascal, polarity, picks in selections:
        for priority, fk in picks:
            if fk not in ground_truth_entries:
                if fk not in yaml_cache:
                    yaml_cache[fk] = read_yaml(fk)
                if yaml_cache[fk] is None:
                    skipped_no_yaml.append(fk)
                    continue
                ground_truth_entries[fk] = _build_entry(
                    rows_by_key[fk], yaml_cache[fk], smells_by_key[fk]
                )
            existing = {
                (t["pascal"], t["polarity"])
                for t in ground_truth_entries[fk]["evaluation_targets"]
            }
            if (pascal, polarity) not in existing:
                ground_truth_entries[fk]["evaluation_targets"].append({
                    "pascal": pascal,
                    "polarity": polarity,
                    "priority": priority,
                })

    # Deterministic ordering: paper_id ASC then polarity, file_key ASC.
    for entry in ground_truth_entries.values():
        entry["evaluation_targets"].sort(
            key=lambda t: (SMELL_CATALOG[t["pascal"]][0], t["polarity"])
        )
    ground_truth_list = [ground_truth_entries[k] for k in sorted(ground_truth_entries)]

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(ground_truth_list, f, indent=2, ensure_ascii=False)

    report = {
        "requests": requests,
        "n_resample_per_target": n,
        "exclusion_source": str(prior_ground_truth_path),
        "exclusion_source_sha256": prior_sha,
        "excluded_triples": len(excluded),
        "total_resample_entries": len(ground_truth_list),
        "unique_workflow_files": len(ground_truth_list),
        "skipped_no_yaml": skipped_no_yaml,
        "sampling_config": {
            "algorithm": (
                "targeted top-up: strict file-wide filter, "
                "(file,smell,polarity)-excluded, LOC-capped, quartile-stratified"
            ),
            "strict_filter": "every behavioral smell instance_count <= 1",
            "strict_pool_size": len(strict_keys),
            "rejected_strict": rejected_strict,
            "loc_cap_used": loc_cap,
            "loc_cap_source": str(loc_cap_path),
            "loc_cap_sha256": loc_cap_sha,
            "loc_cap_excluded": loc_cap_excluded,
            "rows_before_loc_cap": rows_before_cap,
            "rows_after_loc_cap": rows_after_cap,
        },
        "per_target": per_target_stats,
        "input_sha256": input_sha,
    }
    if output_path.exists():
        report["output_file"] = str(output_path)
        report["output_sha256"] = compute_file_hash(output_path)
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    return report


# -- Console output ------------------------------------------------------------

def _print_target_table(report: dict) -> None:
    print()
    print(f"  Re-sample coverage (target {report['n_resample_per_target']} fresh per request):")
    print(f"  {'ID':>3}  {'Smell':<52} {'Pol':<4} {'Avail':>7} {'Deliv':>6} {'Short':>6}")
    print(f"  {'---':>3}  {'-' * 52} {'-' * 4} {'-' * 7} {'-' * 6} {'-' * 6}")
    for t in report["per_target"]:
        print(
            f"  {t['paper_id']:>3}  {t['pascal'][:52]:<52} {t['polarity'][:3]:<4}"
            f" {t['available_fresh']:>7,} {t['delivered']:>6} {t['shortfall']:>6}"
        )
    cfg = report["sampling_config"]
    print()
    print(f"  Strict pool      : {cfg['strict_pool_size']:,}")
    print(f"  Excluded triples : {report['excluded_triples']:,}")
    print(f"  Total entries    : {report['total_resample_entries']:,}")
    print(f"  Skipped (no yaml): {len(report['skipped_no_yaml'])}")


report = resample_ground_truth(
    REQUESTS,
    candidates_path=Path("data/9_ground_truth_candidates.csv"),
    prior_ground_truth_path=Path("data/10_ground_truth_sample.json"),
    output_path=Path("data/10b_ground_truth_resample.json"),
    report_path=Path("data/step10b_report.json"),
)

print("Re-sample generated.")
print("  Output : data/10b_ground_truth_resample.json")
print("  Report : data/step10b_report.json")
_print_target_table(report)

Step 10b — starting: targeted ground_truth re-sample (top-up)...


  LOC cap from Step 09b: 209 (sha 3dfeb6a1e48dd19e...)


  exclusion set: 3,800 (file,smell,polarity) triples from 10_ground_truth_sample.json


  reading input CSV (may take a moment for large files)...


  LOC cap applied: 4,716 rows dropped; 45,113 retained.


Re-sample generated.
  Output : data/10b_ground_truth_resample.json
  Report : data/step10b_report.json

  Re-sample coverage (target 100 fresh per request):
   ID  Smell                                                Pol    Avail  Deliv  Short
  ---  ---------------------------------------------------- ---- ------- ------ ------
    2  UnpinnedActionVersion                                neg    9,291    100      0
    5  MissingJobTimeout                                    neg    3,143    100      0
   12  UnsafeDeployOnFork                                   pos   13,386    100      0

  Strict pool      : 14,378
  Excluded triples : 3,800
  Total entries    : 157
  Skipped (no yaml): 0


---